# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

For my lane (**Lane 2: Content Refresh / Opportunity Scoring**), the prediction task is a binary classification problem: *"Will this page's search impressions decline by more than 20% in the next period?"*

I will train two models:
1.  **Logistic Regression** (using the `liblinear` solver to prevent convergence issues on our features). This serves as a baseline model since it is linear, fast, and easily readable.
2.  **Random Forest Classifier**. This fits the lane because decline patterns are non-linear and feature values interact in complex ways (for instance, a high average position is safe only when combined with high CTR, and staleness affects different content types differently). Trees can capture these interactions naturally without manual feature engineering.

In [1]:
# Section 1 — Setup libraries and load dataset
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

# Clean avg_position: 0 is missing data, not rank zero
df['avg_position_clean'] = df['avg_position'].replace(0, np.nan)

# Encode Week-4 baseline score
is_striking = (df['position_tier'] == 'striking').astype(int)
is_stale = (df['days_since_last_update'] >= 90).astype(int)
df['baseline_score'] = is_striking * is_stale * df['impressions_90d']

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a **GroupShuffleSplit grouped by `client_id`** with `test_size=0.25` and `random_state=42`.

A standard random train/test split would allocate different pages from the same client across both train and test partitions. This leaks search volume scale and client-specific template structures, creating a model that memorizes client identities rather than learning general signals of content decline.

Grouping by `client_id` guarantees that the test partition contains entirely unseen clients, which mirrors how the model will perform when deployed on a new client integration.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

# Perform GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))

df_train = df.iloc[train_idx]
df_test = df.iloc[test_idx].copy()

print(f"Grouped Split Design:")
print(f"  Training set: {df_train.shape[0]:,} rows ({df_train['client_id'].nunique()} unique clients)")
print(f"  Testing set:  {df_test.shape[0]:,} rows ({df_test['client_id'].nunique()} unique clients)")

Grouped Split Design:
  Training set: 22,885 rows (24 unique clients)
  Testing set:  7,115 rows (8 unique clients)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I preprocess the features by imputing missing values with the median, standardizing the scale of numeric inputs, and encoding categoricals. I evaluate models on ROC-AUC, Precision@10, and Precision@50.

In [3]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Features selected
numeric_features = ['impressions_90d', 'clicks_90d', 'avg_position_clean', 'days_since_last_update', 'word_count']
categorical_features = ['position_tier', 'content_type', 'main_intent']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Pipelines
lr_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(solver='liblinear', random_state=42))
])

rf_pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
])

# Fit models
lr_pipeline.fit(df_train, df_train['is_declining'])
rf_pipeline.fit(df_train, df_train['is_declining'])

# Predict probabilities
y_test = df_test['is_declining']
base_probs = df_test['baseline_score'].values
lr_probs = lr_pipeline.predict_proba(df_test)[:, 1]
df_test['rf_prob'] = rf_pipeline.predict_proba(df_test)[:, 1]

# Metrics helper
def eval_model(name, probs, y_true):
    auc = roc_auc_score(y_true, probs)
    order = np.argsort(-probs)
    p10 = y_true.values[order[:10]].mean()
    p50 = y_true.values[order[:50]].mean()
    return {'Model': name, 'ROC-AUC': f"{auc:.4f}", 'Precision@10': f"{p10:.2f}", 'Precision@50': f"{p50:.2f}"}

# Print metrics comparison table
results = [
    {'Model': 'Base Rate', 'ROC-AUC': '0.5000', 'Precision@10': f"{y_test.mean():.2f}", 'Precision@50': f"{y_test.mean():.2f}"},
    eval_model('Baseline Rule', base_probs, y_test),
    eval_model('Logistic Regression', lr_probs, y_test),
    eval_model('Random Forest', df_test['rf_prob'].values, y_test)
]
print(pd.DataFrame(results).to_string(index=False))

              Model ROC-AUC Precision@10 Precision@50
          Base Rate  0.5000         0.52         0.52
      Baseline Rule  0.5021         0.30         0.50
Logistic Regression  0.5211         0.50         0.50
      Random Forest  0.5986         0.70         0.74


/opt/homebrew/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/homebrew/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I will review the top feature importances and inspect the largest errors to understand where my Random Forest model breaks.

In [4]:
# Feature importances
rf_clf = rf_pipeline.named_steps['clf']
ohe_cols = rf_pipeline.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(categorical_features)
features = numeric_features + list(ohe_cols)
importances = rf_clf.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)
print("Top 5 Feature Importances:")
print(feat_imp.head(5).to_string())
print()

# 1. False Positives (y_test = 0, high predicted prob)
fps = df_test[(df_test['is_declining'] == 0)].sort_values(by='rf_prob', ascending=False)
print("Top False Positives:")
print(fps[['content_id', 'impressions_90d', 'clicks_90d', 'avg_position_clean', 'days_since_last_update', 'rf_prob']].head(2).to_string(index=False))
print()

# 2. False Negatives (y_test = 1, low predicted prob)
fns = df_test[(df_test['is_declining'] == 1)].sort_values(by='rf_prob', ascending=True)
print("Top False Negatives:")
print(fns[['content_id', 'impressions_90d', 'clicks_90d', 'avg_position_clean', 'days_since_last_update', 'rf_prob']].head(2).to_string(index=False))

Top 5 Feature Importances:
impressions_90d           0.311436
word_count                0.175082
avg_position_clean        0.137325
days_since_last_update    0.094092
clicks_90d                0.089992

Top False Positives:
          content_id  impressions_90d  clicks_90d  avg_position_clean  days_since_last_update  rf_prob
content_3672013e1d63             2488           5                42.3                     104 0.859117
content_1d0963b56227             3445           3                39.0                     104 0.857954

Top False Negatives:
          content_id  impressions_90d  clicks_90d  avg_position_clean  days_since_last_update  rf_prob
content_8c482a64a3df                1           0                 3.0                      20 0.050676
content_7bc32bc1df59                1           0                 NaN                      92 0.064021


### Interpretation of Errors & Features

**Feature Importance Analysis:**
*   `impressions_90d` (31.1%) is the most important feature, followed by `word_count` (17.5%) and `avg_position_clean` (13.7%). This confirms that the model relies heavily on traffic scale and content volume to evaluate stability.

**Concrete Error Analysis:**
1.  **False Positive (`content_3672013e1d63`):** 
    *   *Symptom:* High predicted risk (85.9%) but actual outcome was stable (0).
    *   *Why:* The page had not been updated in 104 days and sits at a deep ranking position (42.3) with poor CTR (5 clicks out of 2,488 impressions). Historically, pages in this condition are highly likely to slide. However, for pages that are already deep in the ranks, traffic levels are so close to zero that they fluctuate randomly without experiencing a true structural decline.
2.  **False Negative (`content_8c482a64a3df`):** 
    *   *Symptom:* Low predicted risk (5.1%) but actual outcome was declining (1).
    *   *Why:* The page is extremely fresh (20 days old) and sits at a very high rank (average position 3.0), which makes the model predict it as completely safe. However, it had only 1 impression in the entire 90-day window. With such small numbers, the decline label is highly volatile and could be triggered by going from 1 impression to 0 (a 100% decline), which represents statistical noise rather than a structural decline.

**Key Takeaway:** The model struggles with low-volume pages because the target label is highly volatile and sensitive to minor random fluctuations.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.